# Espresso Log — analysis

Reads the local SQLite database and looks at four questions:

1. Which bean scored best?
2. Does extraction time predict quality?
3. How does brew ratio relate to the rating?
4. Am I getting better over time?
5. Which grinder setting lands in the target extraction window?

The database is not part of the repository. Create it first:

```bash
python3 src/log_shot.py init
sqlite3 data/espresso.db < seed.sql
```

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# The notebook lives in notebooks/, the database in data/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "espresso.db"

if not DB_PATH.exists():
    raise FileNotFoundError(f"No database at {DB_PATH} — see the cell above.")

connection = sqlite3.connect(DB_PATH)

# v_shot_details is the view from schema.sql: shots joined to beans, with
# brew ratio and days off roast already computed.
shots = pd.read_sql_query(
    "SELECT * FROM v_shot_details ORDER BY shot_date, id",
    connection,
    parse_dates=["shot_date"],
)

print(f"{len(shots)} shots, {shots['bean_name'].nunique()} beans")
shots.head()

## 1. Which bean scored best?

Averaging the rating per bag. Bags with fewer than five shots are dropped —
an average over two shots says very little.

In [ ]:
by_bean = pd.read_sql_query(
    """
    SELECT b.name AS bean,
           COUNT(*) AS shots,
           ROUND(AVG(s.taste_rating), 2) AS avg_rating
      FROM shots AS s
      JOIN beans AS b ON b.id = s.bean_id
     GROUP BY b.id
    HAVING COUNT(*) >= 5
     ORDER BY avg_rating DESC
    """,
    connection,
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(by_bean["bean"], by_bean["avg_rating"], color="#6f4e37")
ax.set_xlabel("average rating (1-5)")
ax.set_xlim(0, 5)
ax.set_title("Average taste rating per bean")
ax.invert_yaxis()

for y, (rating, n) in enumerate(zip(by_bean["avg_rating"], by_bean["shots"])):
    ax.text(rating + 0.05, y, f"{rating}  (n={n})", va="center")

plt.tight_layout()
plt.show()

by_bean

## 2. Does extraction time predict quality?

Each shot plotted as extraction time against rating. The shaded band marks
the 22–32 second window that espresso guidance usually recommends.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.axvspan(22, 32, color="#6f4e37", alpha=0.12, label="target window 22-32s")

for bean, group in shots.groupby("bean_name"):
    ax.scatter(group["extraction_time_s"], group["taste_rating"],
               label=bean, s=60, alpha=0.8)

ax.set_xlabel("extraction time (s)")
ax.set_ylabel("taste rating")
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_title("Extraction time vs. rating")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Brew ratio vs. rating

Brew ratio is yield divided by dose. A classic espresso sits near 1:2.

In [ ]:
rated = shots.dropna(subset=["brew_ratio", "taste_rating"])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(rated["brew_ratio"], rated["taste_rating"],
           s=60, alpha=0.75, color="#6f4e37")
ax.axvline(2.0, linestyle="--", linewidth=1, color="grey",
           label="1:2 reference")

ax.set_xlabel("brew ratio (yield / dose)")
ax.set_ylabel("taste rating")
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_title("Brew ratio vs. rating")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Am I getting better over time?

Individual ratings jump around too much to read a trend from, so a rolling
average over five shots is drawn on top.

In [ ]:
trend = shots.dropna(subset=["taste_rating"]).copy()
trend["rolling_avg_5"] = trend["taste_rating"].rolling(5, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(trend["shot_date"], trend["taste_rating"],
        marker="o", linewidth=0, alpha=0.4, label="single shot")
ax.plot(trend["shot_date"], trend["rolling_avg_5"],
        linewidth=2, color="#6f4e37", label="rolling average (5 shots)")

ax.set_ylabel("taste rating")
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_title("Rating over time")
ax.legend(fontsize=8)
fig.autofmt_xdate()

plt.tight_layout()
plt.show()

## 5. Which grinder setting lands in the target window?

One point per shot. The grind setting is stored as a number, which is what
makes this plot possible at all — as text it could only be counted, not
placed on an axis. The shaded band is the 22-32 second target window.

Both axes are coarse (five grinder steps, whole seconds), so many shots
share the exact same coordinates and would hide each other. A small random
horizontal offset is added to separate them — the seed is fixed so the
picture stays reproducible.

This is the chart embedded in the README, so it is also written to
`assets/grind-vs-extraction.png`.

In [ ]:
import numpy as np

grind = shots.dropna(subset=["grind_setting", "extraction_time_s"])

# Fixed seed: same jitter on every run, so the exported PNG is stable.
rng = np.random.default_rng(0)
jitter = rng.uniform(-0.05, 0.05, len(grind))

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.axhspan(22, 32, color="#6f4e37", alpha=0.12,
           label="target window 22-32s")

scatter = ax.scatter(
    grind["grind_setting"] + jitter,
    grind["extraction_time_s"],
    c=grind["taste_rating"],
    cmap="RdYlGn",
    vmin=1, vmax=5,
    s=110, alpha=0.9, edgecolor="black", linewidth=0.4,
)

ax.set_xlabel("grind setting (finer <-- --> coarser), jittered")
ax.set_ylabel("extraction time (s)")
ax.set_title(f"Grind setting vs. extraction time (n={len(grind)} shots)")
ax.set_xticks(sorted(grind["grind_setting"].unique()))
ax.legend(loc="upper right", fontsize=8)

colourbar = fig.colorbar(scatter, ax=ax)
colourbar.set_label("taste rating")
colourbar.set_ticks([1, 2, 3, 4, 5])

plt.tight_layout()

# Written to disk so the README can embed exactly this figure.
ASSETS = ROOT / "assets"
ASSETS.mkdir(exist_ok=True)
fig.savefig(ASSETS / "grind-vs-extraction.png", dpi=110)

plt.show()

In [ ]:
connection.close()